<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Explain predictions

**[C05-M05-L02 · Explain predictions](https://learning.nextia-ai.com/courses/ml/m05/explain-predictions/)** · C05, Machine Learning: From Problem to Reliable Model · Module 5, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read the coefficients of the final model, measure permutation importance, split one ticket's score into contributions, and say what these explanations do **not** tell you: a reason or a cause.

| | |
|---|---|
| **Time** | About 60 minutes |
| **Needs** | An internet connection for the setup cells. No account. |
| **You should know** | The logistic function and the weighted sum, from [Classification intuition](https://learning.nextia-ai.com/courses/ml/m02/classification-intuition/). Scaling and one-hot encoding, from [Preprocess consistently](https://learning.nextia-ai.com/courses/ml/m02/preprocess-consistently/). Average precision, from [Choose metrics](https://learning.nextia-ai.com/courses/ml/m04/choose-metrics/). |
| **You do not need** | The local project. The setup below downloads the data and the course's `ticket_model.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/explain-predictions/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M05-L02-explain-predictions/explain-predictions-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m01/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. It is the final version from Select useful features, with 11 features. The next cells import it.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A score is a weighted sum

The final model is a logistic regression. For each ticket, it computes a weighted sum of the prepared values, then turns the sum into a score from 0 to 1 with the logistic function. A **coefficient** is the weight of one prepared column.

The numbers are scaled: a value of 1 means "one standard deviation (SD) above the train mean". Two tickets, with the real coefficients of priority (−0.64) and word_count (+0.40):

| | Ticket A | Ticket B |
|---|---|---|
| priority | 1 → (1 − 2.37) ÷ 0.72 = −1.91 | 3 → (3 − 2.37) ÷ 0.72 = +0.88 |
| priority part | −0.64 × −1.91 = **+1.22** | −0.64 × 0.88 = **−0.56** |
| word_count | 300 → (300 − 106.13) ÷ 70.81 = +2.74 | 40 → (40 − 106.13) ÷ 70.81 = −0.93 |
| words part | 0.40 × 2.74 = **+1.10** | 0.40 × −0.93 = **−0.37** |

> **Predict.** Which ticket gets the higher score? Then run the cell, which repeats the table's arithmetic.

In [ ]:
for name, priority_sd, words_sd in [("A", -1.91, 2.74), ("B", 0.88, -0.93)]:
    priority_part, words_part = -0.64 * priority_sd, 0.40 * words_sd
    print(name, "priority part", round(priority_part, 2), "words part", round(words_part, 2),
          "sum", round(priority_part + words_part, 2))

> **Check.** You should see `A priority part 1.22 words part 1.1 sum 2.32` and `B priority part -0.56 words part -0.37 sum -0.94`.

Ticket A (urgent, long message) gets a much higher sum, so a higher score. Priority 1 has a **negative** coefficient but a **positive** part: the value is below the mean, and minus times minus is plus. Read a coefficient together with the value.

## 2. Fit the final model

Run the next cell. It fits the final model on the train set and scores the validation set, as in the earlier lessons.

> **Check.** You should see `fitted on 16939 tickets; scored 1179 validation tickets`.

In [ ]:
from ticket_model import FEATURES, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])
valid["score"] = pipeline.predict_proba(valid[FEATURES])[:, 1]
print("fitted on", len(train), "tickets; scored", len(valid), "validation tickets")

## 3. The coefficients

The next cell lists every coefficient, from the most negative to the most positive. One-hot columns start with `categories__`, and numbers with `numbers__`.

> **Predict.** Which coefficient is the most negative? Then run the cell.

In [ ]:
names = pipeline.named_steps["prepare"].get_feature_names_out()
coefficients = pd.Series(pipeline.named_steps["model"].coef_[0], index=names).sort_values()
print(coefficients.round(2).to_string())

> **Check.** You should see `categories__team_account -1.02` first and `numbers__prior_escalations_90d 0.43` last. `numbers__prior_tickets_90d` is `-0.29`.

## 4. The sign trap: prior_tickets_90d

The coefficient of `prior_tickets_90d` is −0.29: more tickets in the last 90 days **lower** the score. Tomás reads it as "customers who write often are calm". The next cell checks the train data. It shows the escalation rate for four bands of earlier tickets: for all tickets, and only for tickets with no escalation in the 90 days before.

> **Predict.** For all tickets, does the rate go up or down with more earlier tickets? Then run the cell.

In [ ]:
band = pd.cut(train["prior_tickets_90d"], [-1, 0, 2, 5, 1000], labels=["0", "1-2", "3-5", "6 or more"])
none_before = train["prior_escalations_90d"] == 0
print("correlation:", round(train["prior_tickets_90d"].corr(train["prior_escalations_90d"]), 2))
print(pd.DataFrame({
    "all tickets": train.groupby(band, observed=True)[TARGET].mean(),
    "no escalation before": train[none_before].groupby(band[none_before], observed=True)[TARGET].mean(),
}).round(3))

> **Check.** You should see `correlation: 0.73`. For all tickets, the rate goes from 0.104 to 0.137. With no escalation before, it goes from 0.104 to 0.086.

Alone, more earlier tickets go with **more** escalations. But customers with many tickets also have more earlier escalations (correlation 0.73). When `prior_escalations_90d` is **held fixed**, more tickets go with **fewer** escalations. A coefficient answers the second question, not the first. Change the set of features, and the coefficient can change its sign: +0.18 alone, +0.07 with the 10 C04 features, −0.29 in the final model.

## 5. Permutation importance

**Permutation importance** asks: how much worse does the model score if one column is shuffled, so that it carries no information? First, eight tickets and the rule "flag priority 1". Shuffling here means reversing the column's order.

> **Predict.** The rule does not use `word_count`. What happens to its accuracy when you shuffle `word_count`? Then run the cell.

In [ ]:
tiny = pd.DataFrame({"priority": [1, 3, 1, 2, 3, 1, 2, 3],
                     "word_count": [120, 40, 200, 80, 60, 150, 90, 30],
                     "escalated": [1, 0, 1, 0, 0, 0, 0, 0]})


def rule_accuracy(df):
    """The share of tickets where the rule "flag priority 1" is right."""
    return ((df["priority"] == 1).astype(int) == tiny["escalated"]).mean()


print("as recorded:", rule_accuracy(tiny))
print("priority shuffled:", rule_accuracy(tiny.assign(priority=tiny["priority"].values[::-1])))
print("word_count shuffled:", rule_accuracy(tiny.assign(word_count=tiny["word_count"].values[::-1])))

> **Check.** You should see `as recorded: 0.875`, `priority shuffled: 0.625` and `word_count shuffled: 0.875`.

Shuffling `priority` costs 0.875 − 0.625 = 0.25 of accuracy: the rule depends on it. Shuffling `word_count` costs nothing: the rule does not use it. On the real model, scikit-learn shuffles each of the 11 columns 10 times, and measures the average precision lost. It shuffles a category such as `team` as one column, so all its one-hot columns change together. It takes about 10 seconds.

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(pipeline, valid[FEATURES], valid[TARGET], scoring="average_precision",
                                n_repeats=10, random_state=0)
importance = pd.DataFrame({"mean": result.importances_mean, "std": result.importances_std}, index=FEATURES)
print(importance.sort_values("mean", ascending=False).round(3))

> **Check.** You should see `priority 0.111` first, then `prior_escalations_90d 0.075` and `team 0.074`. `customer_tenure_days` is `-0.002`: shuffling it changes nothing. A small negative value is chance.

## 6. One ticket's contributions

For one ticket, the **contribution** of a column is its coefficient times the ticket's prepared value. The intercept plus all contributions is the sum that the logistic function turns into the score. The next cell defines `contributions()` and applies it to T-117033: a priority 3 returns ticket that escalated, but that the model missed with a low score. It shows the columns where the value is not 0.

> **Predict.** Which column pushes this ticket's score down the most? Then run the cell.

In [ ]:
def contributions(pipeline, ticket):
    """Coefficient times prepared value, for each column of one ticket."""
    prepare, model = pipeline.named_steps["prepare"], pipeline.named_steps["model"]
    values = np.asarray(prepare.transform(ticket[FEATURES]))[0]
    table = pd.DataFrame({"value": values, "coefficient": model.coef_[0]},
                         index=prepare.get_feature_names_out())
    table["contribution"] = table["value"] * table["coefficient"]
    return table


ticket = valid[valid["ticket_id"] == "T-117033"]
table = contributions(pipeline, ticket)
print(table[table["value"] != 0].sort_values("contribution").round(2))

> **Check.** You should see `numbers__prior_tickets_90d` first, with a contribution of `-0.59`, and `numbers__prior_escalations_90d` last, with `0.36`.

Run the next cell. It adds the intercept and the contributions, and turns the sum into a score with the logistic function.

In [ ]:
intercept = pipeline.named_steps["model"].intercept_[0]
total = intercept + table["contribution"].sum()
print("intercept", round(intercept, 2), "+ contributions", round(table["contribution"].sum(), 2), "=", round(total, 2))
print("score from the sum:", round(1 / (1 + np.exp(-total)), 3), "| predict_proba:", round(ticket["score"].iloc[0], 3))

> **Check.** You should see `intercept -1.28 + contributions -2.23 = -3.52` and `score from the sum: 0.029 | predict_proba: 0.029`.

The contributions add up exactly to the model's score. But read the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/explain-predictions/#what-a-contribution-is-not) before you call them "reasons": they compare this ticket with the average train ticket, through a model that knows only 11 columns.

## 7. Guided practice: a caught ticket

> **Your turn.** Do the same for T-117014, a priority 1 payment ticket that the model flagged. Put its contributions table in `table2`. Then put the name of the column with the largest contribution in `biggest` (use `idxmax()`), and the score from the sum in `score_from_sum`, rounded to 3 decimals.

In [ ]:
ticket2 = valid[valid["ticket_id"] == "T-117014"]
table2 = ...
biggest = ...
score_from_sum = ...
print(biggest, score_from_sum)

In [ ]:
expect_hash('the largest contribution', biggest, '5e44d31a75c742e6')

In [ ]:
expect_hash('the score from the sum', score_from_sum, '533a26b6cb8cb2c7')

## 8. Change one thing: rank by ROC AUC

Permutation importance depends on the metric you choose. Change one setting: `scoring="roc_auc"` instead of `"average_precision"`.

> **Predict.** Will the order of the top three columns stay the same?

> **Your turn.** Run `permutation_importance` again with `scoring="roc_auc"` and the same other settings. Put the names of the three most important columns, in order, in the list `top3`.

In [ ]:
result_auc = ...
top3 = ...
print(top3)

In [ ]:
expect_hash('the top three columns by ROC AUC', top3, 'eb11e53bd9943980')

Compare the order with section 5, then read the model answer on the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/explain-predictions/#your-turn-rank-by-roc-auc).

## 9. Failure case: importance is not a cause

Priority is the most important feature. Tomás suggests: "Ask agents to set fewer tickets to priority 1. Then the model flags fewer tickets, and fewer escalate." The next cell changes T-117014's priority from 1 to 3 and scores it again.

> **Predict.** What happens to the score? What happens to the customer's problem?

In [ ]:
changed = ticket2.assign(priority=3)
print("score as written:", round(pipeline.predict_proba(ticket2[FEATURES])[0, 1], 3))
print("score with priority 3:", round(pipeline.predict_proba(changed[FEATURES])[0, 1], 3))

> **Check.** You should see `score as written: 0.38` and `score with priority 3: 0.092`.

The score falls below the threshold, so the ticket is no longer flagged. The customer's payment problem is the same, and so is the chance that it needs a senior agent. The model learned that priority goes **with** escalation; it did not learn that priority **causes** it. Changing an input changes the score, not the world.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/explain-predictions/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you read the coefficients of the final model, and saw `prior_tickets_90d` change its sign when `prior_escalations_90d` is held fixed. Permutation importance ranks priority (0.111), prior escalations (0.075) and team (0.074) first. One ticket's contributions add up exactly to its score, but they describe the model, not the reason or the cause of an escalation. Next, you look for groups of customers without a target, with k-means.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Unsupervised exploration](https://learning.nextia-ai.com/courses/ml/m05/unsupervised-exploration/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m05/explain-predictions/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.